# Python-Compatible Interface Design for Standardized MATLAB Outputs

## Overview

The standardized MATLAB dataset generated in the previous stage provided a reusable session-level representation of the acquisition outputs. However, the resulting file was not yet directly suited to downstream Python-based processing and automation because its storage format and nested internal structure were optimized for MATLAB rather than cross-environment use.

This stage therefore focused on designing a Python-compatible interface for the standardized dataset. The objective was not to alter the meaning of the data, but to preserve its analysis-relevant content while reorganizing its structure into a lighter and more accessible form for downstream scripting, visualization, and automation.

## Why a Python-Compatible Interface Was Required

Two technical constraints made interface restructuring necessary.  
First, the standardized dataset had been stored in MATLAB v7.3 format in order to support large structured outputs, but that format could not be accessed through the standard `scipy.io.loadmat()` workflow used in Python.  
Second, the original MATLAB struct contained nested fields that were cumbersome for direct access in Python and inefficient for repeated downstream use.

Under those conditions, Python-based analysis would require format-specific loading logic and repeated navigation through nested structures before even basic signal extraction could begin. The practical requirement of this stage was therefore to remove interface-level friction while preserving the underlying signal arrays, event timestamps, and metadata required for later analysis.

## Interface Restructuring Strategy

The original v7.3 MATLAB dataset was loaded through an HDF5-compatible reader, and only the fields required for downstream use were selected. Core signal channels, event timestamps, and sampling-related variables were extracted from the nested structure, flattened into directly accessible arrays, and reorganized into a simplified output schema.

This restructuring did not redefine the dataset itself. Instead, it established a lighter interface layer through which the same standardized information could be accessed more efficiently in Python. The result was a flat and Python-compatible interface layer that reduced loading overhead, simplified field access, and supported reuse in later scripts and automated analysis.

In [8]:
import numpy as np
from scipy.io import savemat
import hdf5storage
import tkinter as tk
from tkinter import filedialog
import os

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# Select the standardized session-level MATLAB dataset generated in the previous stage
input_path = filedialog.askopenfilename(
    title="Select stimuli.mat file",
    filetypes=[("MATLAB files", "*.mat")]
)
if not input_path:
    raise Exception("No input file selected!")
print("Input:", input_path)

input_dir = os.path.dirname(input_path)
input_name = os.path.basename(input_path)
base_name, _ = os.path.splitext(input_name)
output_path = os.path.join(input_dir, base_name + "_python.mat")

print("Auto-saving as:", output_path)

# Load the MATLAB v7.3 dataset using an HDF5-compatible reader
data = hdf5storage.loadmat(input_path)
stim = data['stimuli']

# Extract the signal arrays, event timestamps, and metadata required for downstream Python-based analysis
photo1 = stim['photo1'][0, 0].flatten()
photo2 = stim['photo2'][0, 0].flatten()
cue1times = stim['cue1times'][0, 0].flatten()
sol1times = stim['sol1times'][0, 0].flatten()
downsamplingrate = stim['downsamplingrate'][0, 0].flatten()[0]

# Flatten nested MATLAB fields into directly accessible Python arrays
stimuli_python = {
    "photo1": photo1.astype(np.float32),
    "photo2": photo2.astype(np.float32),
    "cue1times": cue1times.astype(np.float32),
    "sol1times": sol1times.astype(np.float32),
    "downsamplingrate": np.array([[int(downsamplingrate)]], dtype=np.int32)
}

# Save a lightweight MATLAB file that can be loaded directly through scipy.io.loadmat() in downstream Python scripts
savemat(output_path, stimuli_python)

print("Conversion complete!")
print("Saved:", output_path)


Input: C:/Users/admin/python_project/stimuli_file/gcajr/Flna30_473+561nm_stimuli.mat
Auto-saving as: C:/Users/admin/python_project/stimuli_file/gcajr\Flna30_473+561nm_stimuli_python.mat
Conversion complete!
Saved: C:/Users/admin/python_project/stimuli_file/gcajr\Flna30_473+561nm_stimuli_python.mat


## Compatibility Validation

After conversion, the restructured dataset was loaded again through the standard `scipy.io.loadmat()` workflow. The major arrays could now be accessed directly without HDF5-specific handling or repeated navigation through nested MATLAB fields.

This validation step confirmed that the converted structure was not only lighter in form, but immediately usable in a standard Python environment. In practical terms, the restructuring established a Python-compatible analysis interface while preserving the standardized content of the original MATLAB dataset.

In [ ]:
# Select the converted Python-compatible dataset
# Load it through the standard scipy.io.loadmat() workflow
# Extract the major signal arrays and event timestamps
# Confirm direct access to the restructured interface

from scipy.io import loadmat
import numpy as np
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

mat_path = filedialog.askopenfilename(
    title="Select stimuli_python.mat file",
    filetypes=[("MATLAB files", "*.mat")]
)

if not mat_path:
    raise Exception("No file selected!")

print("Loaded:", mat_path)

data = loadmat(mat_path)

photo1 = data['photo1'].flatten()
photo2 = data['photo2'].flatten()
sol1times = data['sol1times'].flatten()
cue1times = data['cue1times'].flatten()
downsamplingrate = int(data['downsamplingrate'][0][0])

print("Validation load completed.")

## Structure Verification Through Event-Aligned Check

A reward-aligned plotting step was used to verify that the converted dataset preserved the temporal structure required for downstream analysis. Signal segments were extracted relative to reward delivery, baseline-adjusted, and averaged across trials to confirm that the event-aligned organization remained intact after MATLAB-to-Python restructuring.

The purpose of this step was not final biological interpretation, but structural verification. Specifically, it confirmed that the converted interface preserved signal continuity, event timing relationships, and analysis compatibility after transfer into the Python environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the reward-aligned validation window used for structure verification
pre, post = 2, 6

n_trials = len(sol1times)

signal1_segments = []
signal2_segments = []

# Extract reward-aligned signal segments to verify that event timing and signal continuity were preserved after interface conversion
for t in sol1times:
    idx = int(t * downsamplingrate)

    start = idx - int(pre * downsamplingrate)
    end = idx + int(post * downsamplingrate)

    if start >= 0 and end < len(photo1):
        # Use the -3 to -2 s interval as the baseline window for segment normalization
        base_start = idx - int(3 * downsamplingrate)
        base_end = idx - int(2 * downsamplingrate)

        baseline1 = np.mean(photo1[base_start:base_end])
        baseline2 = np.mean(photo2[base_start:base_end])

        segment1 = photo1[start:end] - baseline1
        segment2 = photo2[start:end] - baseline2

        signal1_segments.append(segment1)
        signal2_segments.append(segment2)

signal1_segments = np.array(signal1_segments)
signal2_segments = np.array(signal2_segments)

time_vec = np.linspace(-pre, post, signal1_segments.shape[1])

# Plot trial-averaged reward-aligned traces as a structural validation check
plt.plot(time_vec, signal1_segments.mean(axis=0), label='Signal 1')
plt.plot(time_vec, signal2_segments.mean(axis=0), label='Signal 2')
plt.axvline(0, linestyle='--')
plt.title('Reward-aligned validation plot')
plt.xlabel('Time (s)')
plt.ylabel('Baseline-adjusted intensity')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## Outcome

This stage established the interface layer required to extend the workflow from MATLAB-based standardization to Python-based analysis and automation. By converting a large and nested MATLAB dataset into a lightweight, flat, and accessible structure, the workflow preserved its standardized content while improving downstream usability.

As a result, the project could move forward with a dataset representation that remained faithful to the reconstructed session structure while supporting later visualization, comparison, algorithm validation, and automated analysis in Python.